In [1]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

In [ ]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [ ]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20312\996206371.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])


In [ ]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [ ]:
df = df[df['residual_bond_maturity'] >= 0]

In [ ]:
df['collateral_country'] = df['ISIN'].str[:2]

In [ ]:
df = df[df['collateral_country'].isin(['DE','IT'])]

In [ ]:
securities = tuple(df['ISIN'].unique())

In [ ]:
# Lender side
query = f"""
SELECT s.lender_id AS entity_id, s.lender_name AS name,
       COUNT(*) AS n, SUM(nominal_euro) AS volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY s.lender_id, s.lender_name
"""
df_hf = pd.read_sql_query(query, cnxn)
vol_l = df_hf.groupby('entity_id', as_index=False)['volume'].sum().rename(columns={'volume': 'volume_lender'})
df_hf = (df_hf.sort_values(['entity_id', 'n', 'name'], ascending=[True, False, True])
              .drop_duplicates('entity_id')
              .rename(columns={'name': 'lender_name'})
              .drop(columns=['n', 'volume'])
              .merge(vol_l, on='entity_id'))

# Borrower side
query = f"""
SELECT s.borrower_id AS entity_id, s.borrower_name AS name,
       COUNT(*) AS n, SUM(nominal_euro) AS volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY s.borrower_id, s.borrower_name
"""
df_hf_b = pd.read_sql_query(query, cnxn)
vol_b = df_hf_b.groupby('entity_id', as_index=False)['volume'].sum().rename(columns={'volume': 'volume_borrower'})
df_hf_b = (df_hf_b.sort_values(['entity_id', 'n', 'name'], ascending=[True, False, True])
                  .drop_duplicates('entity_id')
                  .rename(columns={'name': 'borrower_name'})
                  .drop(columns=['n', 'volume'])
                  .merge(vol_b, on='entity_id'))

# Combine
df_hf = df_hf.merge(df_hf_b, on='entity_id', how='outer')
df_hf['name'] = df_hf['lender_name'].fillna(df_hf['borrower_name'])
df_hf['volume'] = df_hf['volume_lender'].fillna(0) + df_hf['volume_borrower'].fillna(0)
df_hf = (df_hf[['entity_id', 'name', 'volume_lender', 'volume_borrower', 'volume']]
             .sort_values('volume', ascending=False)
             .reset_index(drop=True))

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20312\843515743.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf = pd.read_sql_query(query, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_20312\843515743.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf_b = pd.read_sql_query(query, cnxn)
